# MGVS Kaggle Submission Starter

Thin offline runner: install package wheel from attached dataset, load config, run solver, write deterministic submission.

In [ ]:
import glob
import json
import os
import subprocess
import sys

MGVS_DATASET_SLUG = "<your-mgvs-dataset-slug>"
BUNDLE_ROOT = f"/kaggle/input/{MGVS_DATASET_SLUG}/kaggle_bundle"
wheel_candidates = sorted(glob.glob(f"{BUNDLE_ROOT}/wheels/*.whl"))
if not wheel_candidates:
    raise FileNotFoundError("No mgvs wheel found. Check attached dataset slug/path.")

subprocess.check_call([sys.executable, "-m", "pip", "install", "--no-deps", wheel_candidates[0]])
print("Installed wheel:", wheel_candidates[0])

config_path = f"{BUNDLE_ROOT}/config/runtime_config.json"
if os.path.exists(config_path):
    with open(config_path, "r", encoding="utf-8") as handle:
        RUNTIME_CONFIG = json.load(handle)
else:
    RUNTIME_CONFIG = {}
print("Loaded runtime config keys:", sorted(RUNTIME_CONFIG.keys()))

In [ ]:
from mgvs.solve.runner import SolveConfig, solve

BACKEND = str(RUNTIME_CONFIG.get("backend", "stub"))
CONFIG = SolveConfig(
    target_type=str(RUNTIME_CONFIG.get("target_type", "competition")),
    max_depth=int(RUNTIME_CONFIG.get("max_depth", 4)),
    beam_width=int(RUNTIME_CONFIG.get("beam_width", 3)),
    max_candidates=int(RUNTIME_CONFIG.get("max_candidates", 3)),
)

ENABLE_SMOKE_TEST = bool(RUNTIME_CONFIG.get("enable_smoke_test", False))
if ENABLE_SMOKE_TEST:
    smoke = solve("If x = 1, compute x.", config=CONFIG)
    print("Smoke status:", smoke.best_state.status.value, "answer:", smoke.predicted_answer)


In [ ]:
import pandas as pd

COMPETITION_INPUT = "/kaggle/input/<competition-slug>/test.csv"
submission_cfg = RUNTIME_CONFIG.get("submission", {})
ID_COL = str(submission_cfg.get("id_col", "id"))
PROBLEM_COL = str(submission_cfg.get("problem_col", "problem"))
ANSWER_COL = str(submission_cfg.get("answer_col", "answer"))

test_df = pd.read_csv(COMPETITION_INPUT)
test_df.head()

In [ ]:
rows = []
for row in test_df.itertuples(index=False):
    row_dict = row._asdict()
    row_id = row_dict.get(ID_COL)
    problem = str(row_dict.get(PROBLEM_COL, "")).strip()

    result = solve(problem, config=CONFIG)
    prediction = "" if result.predicted_answer is None else str(result.predicted_answer)
    rows.append({ID_COL: row_id, ANSWER_COL: prediction})

submission_df = pd.DataFrame(rows)
submission_df = submission_df.sort_values(by=ID_COL, kind="stable")
submission_df.to_csv("submission.csv", index=False)
submission_df.head()

In [ ]:
print("Wrote submission.csv")